# 🛰️ Mapping Natural-Disaster Damage from Satellite Imagery

**Team:** Bhuvanesh Dinesh Wadhwani · Artur Zavistovskyi · Anastasiia Khitrova · Yitian Zhou

This is a **single self-contained notebook** — everything (data, model code, training,
evaluation) lives here, because Kaggle lets you open only one notebook. It runs the whole
story end-to-end:

> **Part 1** — data + EDA → **Part 2** — two CNNs + held-out evaluation → **Part 3** — impact analysis

---

## 🎯 What we build
We take satellite photos **before** and **after** a disaster and train a network to look at the
*same building in both photos* and rate how badly it was damaged on a 4-level scale:

`no-damage → minor-damage → major-damage → destroyed`

Comparing before vs after is called **change detection**.

### 🔬 The research question
> *If the model learns hurricane / flood / earthquake damage, can it recognise **wildfire**
> damage it has never seen?*

Damage looks very different across disasters, so we **train on some disaster types and test on a
completely different held-out one (wildfire)** — not a random split. That transfer gap is the
heart of the project. We also build **two** models (from-scratch vs pretrained) and compare them.

---

## 🔌 One switch controls everything: `USE_SYNTHETIC`
| Value | What happens | When |
|-------|--------------|------|
| `True` | Generates toy "buildings before/after" in-notebook. Full pipeline in ~3–6 min, **no data needed**. Numbers are fake. | Learning / developing / quick check |
| `False` | Loads the real **xBD / xView2** data you attached to the notebook. Real results. | The actual run |

Same models, same code — only the data source changes.

## 🧭 How to run on Kaggle
1. **Add Data** ▸ attach your xBD dataset (the 4 uploaded `.tar` files).
2. Set `USE_SYNTHETIC = False` and confirm the path in the **Data path** cell (or leave auto-detect).
3. **Run All**.

> ⚠️ With synthetic data the accuracy numbers are toy numbers — they only prove the machinery
> works. Real conclusions come from the xBD run.

## 0 · Setup & the master switch
Imports, a reproducibility seed, the compute device, and the one switch that picks
synthetic vs real data.

In [ ]:
import os, sys, time, json, re, collections, tarfile
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader

# ⬇️⬇️⬇️  THE ONE SWITCH  ⬇️⬇️⬇️
USE_SYNTHETIC = True      # True = toy data, no download.  False = real xBD (attach it first).
# ⬆️⬆️⬆️                  ⬆️⬆️⬆️

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Mode   :', 'SYNTHETIC (quick test)' if USE_SYNTHETIC else 'REAL xBD')
print('Device :', device)

## ⚙️ Configuration
All knobs in one place: the 4 damage classes, the **cross-disaster split** (train on 3 disaster
types, hold out wildfire for testing), patch size and training hyper-parameters. xBD event names
encode the disaster type by keyword (e.g. `santa-rosa-wildfire` → `wildfire`), so we map by keyword.

In [ ]:
# --- damage labels (xBD 4-level scale) ---
DAMAGE_CLASSES = ["no-damage", "minor-damage", "major-damage", "destroyed"]
NAME_TO_IDX = {name: i for i, name in enumerate(DAMAGE_CLASSES)}
NUM_CLASSES = len(DAMAGE_CLASSES)

# --- cross-disaster split (NOT random — this is the graded design) ---
TRAIN_DISASTERS = ["hurricane", "flood", "earthquake"]   # train / val on these
HELDOUT_DISASTER = "wildfire"                             # hold out entirely for test

# --- patches & training ---
PATCH_SIZE = 64      # px; small keeps training fast
BATCH_SIZE = 64
EPOCHS = 5
LR = 1e-3

_DISASTER_KEYWORDS = {
    "hurricane": "hurricane", "flood": "flood",
    "fire": "wildfire", "wildfire": "wildfire", "bushfire": "wildfire",
    "earthquake": "earthquake", "tsunami": "tsunami",
    "volcano": "volcano", "tornado": "tornado",
}

def event_to_disaster_type(event_name):
    # 'santa-rosa-wildfire' -> 'wildfire',  'hurricane-florence' -> 'hurricane'
    name = event_name.lower()
    for keyword, dtype in _DISASTER_KEYWORDS.items():
        if keyword in name:
            return dtype
    return "other"

print('Train on:', TRAIN_DISASTERS, '| Hold out for test:', HELDOUT_DISASTER)

## 📂 Data path (Kaggle)
On Kaggle your attached dataset is mounted **read-only** at `/kaggle/input/<slug>/`; Kaggle
auto-extracts the `.tar` archives, so `train/ test/ tier3/` appear as folders. This cell finds
those source roots and prepares a **writable** working dir (`/kaggle/working`) for the patches and
checkpoints we generate. Leave `DATA_PATH = None` to auto-detect, or paste the path.

In [ ]:
DATA_PATH = None      # 👈 paste e.g. "/kaggle/input/xbd-xview2"  (or leave None to auto-detect)

# writable scratch (patches / manifest / checkpoints) — /kaggle/input is READ-ONLY
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("work")
PROCESSED_DIR = WORK_DIR / "processed"
CKPT_DIR = WORK_DIR / "checkpoints"
for d in (PROCESSED_DIR, CKPT_DIR):
    d.mkdir(parents=True, exist_ok=True)

def _autodetect_root():
    kin = Path("/kaggle/input")
    if DATA_PATH:
        return Path(DATA_PATH)
    if kin.is_dir():
        subs = [p for p in kin.iterdir() if p.is_dir()]
        return subs[0] if len(subs) == 1 else kin      # one dataset -> use it directly
    return Path("data")                                 # local default

def find_xbd_sources(root):
    # any folder holding BOTH images/ and labels/ is an xBD source root
    root = Path(root)
    roots = set()
    if (root / "images").is_dir() and (root / "labels").is_dir():
        roots.add(root)
    for lbl in root.rglob("labels"):                    # train/ test/ tier3/ at any depth
        if lbl.is_dir() and (lbl.parent / "images").is_dir():
            roots.add(lbl.parent)
    return sorted(roots)

def _maybe_extract_tars(root):
    # fallback: if archives were NOT auto-extracted, unpack once into WORK_DIR
    tars = (list(Path(root).glob("*.tar")) + list(Path(root).glob("*.tgz"))
            + list(Path(root).glob("*.tar.gz")))
    if not tars:
        return None
    dest = WORK_DIR / "xbd"; dest.mkdir(parents=True, exist_ok=True)
    for t in tars:
        marker = dest / (t.name + ".done")
        if marker.exists():
            continue
        print(f"⏳ extracting {t.name} … (first run only)")
        with tarfile.open(t) as tf:
            tf.extractall(dest)
        marker.touch()
    return dest

ROOT = _autodetect_root()
SOURCES = find_xbd_sources(ROOT)
if not SOURCES and not USE_SYNTHETIC:
    ex = _maybe_extract_tars(ROOT)
    if ex:
        SOURCES = find_xbd_sources(ex)

print("📁 dataset root :", ROOT)
if SOURCES:
    print("✅ xBD sources  :")
    for s in SOURCES:
        n = len(list((s / "labels").glob("*.json")))
        print(f"     • {s}  ({n} label files)")
else:
    msg = "(synthetic mode — real data not needed)" if USE_SYNTHETIC else \
          "❌ not found — set DATA_PATH to the folder containing train/ test/ tier3/"
    print("ℹ️ xBD sources  :", msg)

_geo = list(Path(ROOT).rglob("*geotransforms*.json")) if Path(ROOT).exists() else []
GEO_PATH = _geo[0] if _geo else None
print("🌍 geotransforms:", GEO_PATH or "not found (only needed for Part 3)")

# Part 1 · Data Collection & Preliminary Analysis  *(10 pts)*
We classify damage **per building** by comparing a `pre` and `post` patch. The held-out disaster
(**wildfire**) is kept entirely for testing — that's the cross-disaster transfer question.
Graders care about: co-registration, handling **class imbalance**, and per-disaster breakdowns.

### 🧩 The data layer (inlined) — A. synthetic generator
Two data sources behind one interface. Both yield `(pre, post, label)` normalized `CxHxW` tensors
and expose `.labels` (for class weights / EDA) and `.raw(i)` (un-normalized, for plotting).

**A — synthetic:** toy "buildings before/after". A disaster "signature" (colour) plus a damage
fraction (how many roof pixels change) let the whole pipeline run with **no download**.

In [ ]:
# ImageNet normalization (the pretrained ResNet expects this).
_MEAN = np.array([0.485, 0.456, 0.406], np.float32)
_STD  = np.array([0.229, 0.224, 0.225], np.float32)

def normalize_chw(hwc):
    # HxWx3 in [0,1] -> normalized 3xHxW tensor
    arr = (hwc - _MEAN) / _STD
    return torch.from_numpy(arr.transpose(2, 0, 1).copy()).float()

def denormalize(hwc):
    return np.clip(hwc, 0, 1)

# --- synthetic "buildings" ---
DISASTER_COLOR = {
    "hurricane":  [0.50, 0.50, 0.55], "flood": [0.20, 0.30, 0.80],
    "earthquake": [0.45, 0.42, 0.40], "wildfire": [0.75, 0.35, 0.08],
    "tsunami":    [0.25, 0.45, 0.70], "tornado": [0.55, 0.52, 0.50],
    "volcano":    [0.30, 0.25, 0.25],
}
DAMAGE_FRAC = {0: 0.0, 1: 0.20, 2: 0.50, 3: 0.90}   # fraction of building pixels changed
CLASS_P     = [0.55, 0.20, 0.15, 0.10]              # class imbalance (most buildings are fine)

def make_pair(damage, disaster, rng, size=None):
    # return (pre, post) toy images, HxWx3 in [0,1]
    size = size or PATCH_SIZE
    base = np.ones((size, size, 3), np.float32) * np.array([0.55, 0.60, 0.45], np.float32)
    base += rng.normal(0, 0.04, base.shape).astype(np.float32)
    h, w = rng.integers(size // 3, size // 2, size=2)
    y, x = int(rng.integers(0, size - h)), int(rng.integers(0, size - w))
    roof = np.array([0.60, 0.60, 0.62], np.float32) + rng.normal(0, 0.03, 3).astype(np.float32)
    pre = base.copy(); pre[y:y+h, x:x+w] = roof
    post = pre.copy()
    mask = rng.random((size, size)) < DAMAGE_FRAC[damage]
    region = np.zeros((size, size), bool); region[y:y+h, x:x+w] = True
    mask &= region
    sig = np.array(DISASTER_COLOR.get(disaster, [0.5, 0.5, 0.5]), np.float32)
    post[mask] = sig + rng.normal(0, 0.05, (int(mask.sum()), 3)).astype(np.float32)
    return np.clip(pre, 0, 1), np.clip(post, 0, 1)

class SyntheticDataset(Dataset):
    def __init__(self, n, disasters, seed):
        rng = np.random.default_rng(seed)
        pre, post, labels = [], [], []
        for _ in range(n):
            d = int(rng.choice(4, p=CLASS_P))
            dis = str(rng.choice(disasters))
            a, b = make_pair(d, dis, rng)
            pre.append(a); post.append(b); labels.append(d)
        self.pre = np.array(pre, np.float32); self.post = np.array(post, np.float32)
        self.labels = np.array(labels)
    def __len__(self):  return len(self.labels)
    def __getitem__(self, i):
        return normalize_chw(self.pre[i]), normalize_chw(self.post[i]), int(self.labels[i])
    def raw(self, i):   return self.pre[i], self.post[i]

def make_synthetic_datasets(n_train=900, n_val=200, n_test=300, seed=SEED):
    # train/val on TRAIN_DISASTERS, test on the held-out disaster only
    train = SyntheticDataset(n_train, TRAIN_DISASTERS, seed)
    val   = SyntheticDataset(n_val,   TRAIN_DISASTERS, seed + 1)
    test  = SyntheticDataset(n_test,  [HELDOUT_DISASTER], seed + 2)
    return train, val, test

### 🧩 The data layer (inlined) — B. real xBD loader
Parse the downloaded tiles into building patches + a `manifest.csv`. For each
`*_post_disaster.json` we read building polygons, crop the **same box** from the pre & post PNGs
(co-registration), drop cloud/blank patches, and label by `subtype`. We re-split **by disaster
type**: the held-out disaster → `test`, everything else → `train` (with a stratified `val` slice).

In [ ]:
_WKT_NUM = re.compile(r"-?\d+\.?\d*")

def _bbox(wkt_str):
    # bounding box from a WKT polygon 'POLYGON ((x1 y1, x2 y2, ...))'; numbers alternate x,y
    nums = [float(v) for v in _WKT_NUM.findall(wkt_str)]
    xs, ys = nums[0::2], nums[1::2]
    return int(min(xs)), int(min(ys)), int(max(xs)), int(max(ys))

def _crop(img, bbox, pad=10, size=None):
    size = size or PATCH_SIZE
    minx, miny, maxx, maxy = bbox
    return img.crop((minx - pad, miny - pad, maxx + pad, maxy + pad)).resize((size, size))

def _is_bad(patch_np, white_frac=0.5):
    # drop near-white (cloud) or near-black (missing) patches
    gray = patch_np.mean(axis=-1)
    return (gray > 240).mean() > white_frac or (gray < 15).mean() > white_frac

def build_xbd_manifest(source_dirs=None, out_dir=None, val_frac=0.15, limit_tiles=500):
    # turn xView2 tiles into building patches + manifest.csv (pre,post,label,disaster,split)
    from PIL import Image
    import pandas as pd
    out = Path(out_dir or PROCESSED_DIR); (out / "patches").mkdir(parents=True, exist_ok=True)
    roots = [Path(r) for r in (source_dirs or SOURCES)]
    if not roots:
        raise FileNotFoundError("No xBD sources. Attach the dataset and check the Data path cell.")
    keep = set(TRAIN_DISASTERS) | {HELDOUT_DISASTER}
    label_files = []
    for root in roots:
        for jp in sorted((root / "labels").glob("*post_disaster.json")):
            if event_to_disaster_type(jp.stem.split("_")[0]) in keep:
                label_files.append((jp, root / "images"))
    if limit_tiles:
        rng_pick = np.random.default_rng(SEED)       # shuffle so a subset still covers all types
        rng_pick.shuffle(label_files)
        label_files = label_files[:limit_tiles]
    print(f"scanning {len(label_files)} tiles from {len(roots)} source(s) …")

    rows = []
    for jp, images_dir in label_files:
        event = jp.stem.split("_")[0]
        dtype = event_to_disaster_type(event)
        post_png = images_dir / (jp.stem + ".png")
        pre_png  = images_dir / (jp.stem.replace("post", "pre") + ".png")
        if not post_png.exists() or not pre_png.exists():
            continue
        pre_img  = Image.open(pre_png).convert("RGB")
        post_img = Image.open(post_png).convert("RGB")
        data = json.loads(jp.read_text())
        for k, feat in enumerate(data["features"].get("xy", [])):
            sub = feat["properties"].get("subtype", "un-classified")
            if sub not in NAME_TO_IDX:                # skip 'un-classified'
                continue
            bbox = _bbox(feat["wkt"])
            pre_p, post_p = _crop(pre_img, bbox), _crop(post_img, bbox)
            if _is_bad(np.asarray(post_p)):
                continue
            stem = f"{jp.stem}_{k}"
            pre_path  = out / "patches" / f"{stem}_pre.png"
            post_path = out / "patches" / f"{stem}_post.png"
            pre_p.save(pre_path); post_p.save(post_path)
            rows.append({"pre_path": str(pre_path), "post_path": str(post_path),
                         "label": NAME_TO_IDX[sub], "disaster_type": dtype,
                         "split": "test" if dtype == HELDOUT_DISASTER else "train"})

    df = pd.DataFrame(rows)
    train_part = df[df["split"] == "train"]           # carve a stratified val slice out of train
    val_idx = (train_part.groupby("label", group_keys=False)
               .sample(frac=val_frac, random_state=SEED).index)
    df.loc[val_idx, "split"] = "val"
    df.to_csv(out / "manifest.csv", index=False)
    print(f"built manifest: {len(df)} patches ->", out / "manifest.csv")
    print(df["split"].value_counts())
    return df

class DamageDataset(Dataset):
    # real-data Dataset reading manifest.csv (one split)
    def __init__(self, split="train", augment=False, manifest_path=None):
        from PIL import Image
        import pandas as pd
        self._Image = Image
        manifest_path = Path(manifest_path or (PROCESSED_DIR / "manifest.csv"))
        if not manifest_path.exists():
            raise FileNotFoundError(f"{manifest_path} not found — run build_xbd_manifest() first.")
        df = pd.read_csv(manifest_path)
        self.df = df[df["split"] == split].reset_index(drop=True)
        self.labels = self.df["label"].to_numpy()
        self.augment = augment
    def __len__(self):  return len(self.df)
    def _load(self, path):
        return np.asarray(self._Image.open(path).convert("RGB").resize(
            (PATCH_SIZE, PATCH_SIZE)), np.float32) / 255.0
    def __getitem__(self, i):
        row = self.df.iloc[i]
        pre, post = self._load(row["pre_path"]), self._load(row["post_path"])
        if self.augment and np.random.rand() < 0.5:
            pre, post = pre[:, ::-1], post[:, ::-1]
        return normalize_chw(pre), normalize_chw(post), int(row["label"])
    def raw(self, i):
        row = self.df.iloc[i]
        return self._load(row["pre_path"]), self._load(row["post_path"])

### 🔍 Data validation — what's actually in the dataset?
Sanity-check the **raw xView2 tiles** before modeling: which disaster types we have, and how many
buildings fall in each damage class. (Runs on real data; skipped in synthetic mode.)

In [ ]:
if not USE_SYNTHETIC and SOURCES:
    tiles_by_disaster = collections.Counter()
    buildings_by_class = collections.Counter()
    for root in SOURCES:
        for jp in (root / 'labels').glob('*post_disaster.json'):
            dtype = event_to_disaster_type(jp.stem.split('_')[0])
            tiles_by_disaster[dtype] += 1
            feats = json.loads(jp.read_text())['features'].get('xy', [])
            for f in feats:
                buildings_by_class[f['properties'].get('subtype', 'un-classified')] += 1
    print(f"Scanned {sum(tiles_by_disaster.values())} post-disaster tiles\n")
    print('Tiles per disaster type (held-out =', HELDOUT_DISASTER, '):')
    for k, v in tiles_by_disaster.most_common():
        flag = '  <- HELD-OUT (test)' if k == HELDOUT_DISASTER else ''
        print(f'  {k:12s} {v:5d}{flag}')
    print('\nBuildings per damage class:')
    total = sum(buildings_by_class.values())
    for k in DAMAGE_CLASSES + ['un-classified']:
        v = buildings_by_class.get(k, 0)
        print(f'  {k:14s} {v:7d}  ({100*v/max(total,1):4.1f}%)')
else:
    print('Skipped (synthetic mode).' if USE_SYNTHETIC else
          'No sources found — check the Data path cell.')

### 🖼️ Peek at real PRE/POST tiles
One example tile per disaster type, so you *see* what the model gets. (Real data only.)

In [ ]:
if not USE_SYNTHETIC and SOURCES:
    from PIL import Image
    samples = {}
    for root in SOURCES:
        for jp in sorted((root / 'labels').glob('*post_disaster.json')):
            dtype = event_to_disaster_type(jp.stem.split('_')[0])
            if dtype not in samples:
                pre = root / 'images' / (jp.stem.replace('post', 'pre') + '.png')
                post = root / 'images' / (jp.stem + '.png')
                if pre.exists() and post.exists():
                    samples[dtype] = (pre, post)
        if len(samples) >= 4:
            break
    n = len(samples)
    fig, axes = plt.subplots(n, 2, figsize=(7, 3.4 * n)); axes = np.array(axes).reshape(n, 2)
    for r, (dtype, (pre, post)) in enumerate(samples.items()):
        axes[r, 0].imshow(Image.open(pre));  axes[r, 0].set_title(f'{dtype} · PRE')
        axes[r, 1].imshow(Image.open(post)); axes[r, 1].set_title(f'{dtype} · POST')
    for ax in axes.ravel(): ax.axis('off')
    plt.tight_layout(); plt.show()
else:
    print('Skipped (synthetic mode).')

### 📦 Load the dataset
Synthetic mode builds toy datasets instantly. Real mode crops building patches on the **first run**
(`limit_tiles=500` keeps it fast and still covers every disaster type via shuffling — set it to
`None` for the full dataset) and caches a `manifest.csv`.

In [ ]:
if USE_SYNTHETIC:
    train_ds, val_ds, test_ds = make_synthetic_datasets()
else:
    manifest = PROCESSED_DIR / "manifest.csv"
    if not manifest.exists():
        build_xbd_manifest(source_dirs=SOURCES, out_dir=PROCESSED_DIR, limit_tiles=500)
    train_ds = DamageDataset('train', augment=True)
    val_ds   = DamageDataset('val',   augment=False)
    test_ds  = DamageDataset('test',  augment=False)

print(f'train {len(train_ds)} | val {len(val_ds)} | test/held-out {len(test_ds)}')
print('Class counts in train:',
      dict(zip(DAMAGE_CLASSES, np.bincount(train_ds.labels, minlength=4))))

### ⚖️ Class distribution
Most buildings are undamaged — this imbalance is why we use **class weights** during training.

In [ ]:
counts = np.bincount(train_ds.labels, minlength=4)
plt.figure(figsize=(6, 3))
plt.bar(DAMAGE_CLASSES, counts, color=['#2ca02c', '#ffdf00', '#ff7f0e', '#d62728'])
plt.title('Damage class distribution (train)'); plt.ylabel('patches'); plt.xticks(rotation=20)
plt.tight_layout(); plt.show()

### 🖼️ Example pre/post patches
Top row = **before**, bottom row = **after**. Damage grows left → right.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(11, 5.5))
for col, d in enumerate(range(4)):
    idx = np.where(train_ds.labels == d)[0]
    if len(idx) == 0:
        continue
    pre, post = train_ds.raw(idx[0])
    axes[0, col].imshow(denormalize(pre));  axes[0, col].set_title(f'PRE · {DAMAGE_CLASSES[d]}')
    axes[1, col].imshow(denormalize(post)); axes[1, col].set_title(f'POST · {DAMAGE_CLASSES[d]}')
for ax in axes.ravel(): ax.axis('off')
plt.tight_layout(); plt.show()

# Part 2 · Deep Learning Models  *(10 pts)*
Two **siamese (two-branch, shared-weight)** change-detection CNNs, compared fairly:
1. **from scratch** — a compact custom CNN
2. **pretrained** — a fine-tuned ResNet18 backbone

Both combine the two branch features as `[pre, post, |pre − post|]` (the abs-difference is the
*change* signal) and predict the 4 damage classes.

### 🏗️ The two architectures (inlined)
Shared-weight encoders so the *same* filters see pre & post; only the comparison head differs.
The pretrained model **falls back to random init** if Kaggle has no internet (enable
*Settings ▸ Internet* for true transfer learning).

In [ ]:
import torch.nn as nn
try:
    from torchvision.models import resnet18, ResNet18_Weights
except Exception:
    resnet18 = None

def _combine(f1, f2):
    # concat both branch features + their absolute difference (the change signal)
    return torch.cat([f1, f2, torch.abs(f1 - f2)], dim=1)

class SiameseScratchCNN(nn.Module):
    # compact two-branch CNN with SHARED weights, built from scratch
    def __init__(self, num_classes=4, feat_dim=128):
        super().__init__()
        def block(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(),
                nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(),
                nn.MaxPool2d(2))
        self.encoder = nn.Sequential(
            block(3, 32), block(32, 64), block(64, 128),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(128, feat_dim), nn.ReLU())
        self.head = nn.Sequential(
            nn.Linear(feat_dim * 3, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, num_classes))
    def forward(self, pre, post):
        return self.head(_combine(self.encoder(pre), self.encoder(post)))

class SiamesePretrainedCNN(nn.Module):
    # two-branch model on a SHARED pretrained ResNet18 backbone, fine-tuned
    def __init__(self, num_classes=4, freeze_until=6):
        super().__init__()
        if resnet18 is None:
            raise ImportError("torchvision is required: pip install torchvision")
        try:
            net = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
            print("  ✓ loaded ImageNet-pretrained ResNet18")
        except Exception:
            net = resnet18(weights=None)
            print("  ⚠ no internet — random-init ResNet18 (enable Kaggle internet for transfer learning)")
        feat_dim = net.fc.in_features
        net.fc = nn.Identity()
        self.encoder = net
        for i, child in enumerate(self.encoder.children()):   # freeze early layers
            if i < freeze_until:
                for p in child.parameters():
                    p.requires_grad = False
        self.head = nn.Sequential(
            nn.Linear(feat_dim * 3, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, num_classes))
    def forward(self, pre, post):
        return self.head(_combine(self.encoder(pre), self.encoder(post)))

### 🔁 Training loop + metrics (inlined)
A generic loop for the `(pre, post, label)` siamese models, plus confusion-matrix / accuracy /
macro-F1 helpers (numpy only, no sklearn). The best-by-val-accuracy checkpoint is saved to
`WORK_DIR/checkpoints`.

In [ ]:
def train_model(model, train_dl, val_dl, device, epochs=EPOCHS, lr=LR,
                class_weights=None, ckpt_path=None, verbose=True):
    model = model.to(device)
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    w = class_weights.to(device) if class_weights is not None else None
    crit = nn.CrossEntropyLoss(weight=w)
    history = {"train_loss": [], "val_loss": [], "val_acc": []}; best = 0.0
    for ep in range(epochs):
        model.train(); tl = 0.0
        for pre, post, y in train_dl:
            pre, post, y = pre.to(device), post.to(device), y.to(device)
            opt.zero_grad()
            loss = crit(model(pre, post), y)
            loss.backward(); opt.step()
            tl += loss.item() * len(y)
        vl, vacc = _validate(model, val_dl, device, crit)
        history["train_loss"].append(tl / len(train_dl.dataset))
        history["val_loss"].append(vl); history["val_acc"].append(vacc)
        if verbose:
            print(f"  epoch {ep+1:02d}/{epochs}  train_loss {history['train_loss'][-1]:.3f}"
                  f"  val_loss {vl:.3f}  val_acc {vacc:.2f}")
        if ckpt_path and vacc > best:
            best = vacc
            Path(ckpt_path).parent.mkdir(parents=True, exist_ok=True)
            torch.save(model.state_dict(), ckpt_path)
    return history

@torch.no_grad()
def _validate(model, dl, device, crit):
    model.eval(); loss = 0.0; correct = 0; n = 0
    for pre, post, y in dl:
        pre, post, y = pre.to(device), post.to(device), y.to(device)
        out = model(pre, post)
        loss += crit(out, y).item() * len(y)
        correct += (out.argmax(1) == y).sum().item(); n += len(y)
    return loss / max(n, 1), correct / max(n, 1)

@torch.no_grad()
def predict(model, dl, device):
    model.eval(); ys, ps = [], []
    for pre, post, y in dl:
        out = model(pre.to(device), post.to(device))
        ys += y.tolist(); ps += out.argmax(1).cpu().tolist()
    return np.array(ys), np.array(ps)

def scores(y_true, y_pred, n_classes=4):
    # -> (confusion_matrix, accuracy, macro_f1, per_class_f1)
    cm = np.zeros((n_classes, n_classes), int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    acc = np.trace(cm) / max(cm.sum(), 1)
    f1s = []
    for c in range(n_classes):
        tp = cm[c, c]
        prec = tp / max(cm[:, c].sum(), 1)
        rec  = tp / max(cm[c, :].sum(), 1)
        f1s.append(2 * prec * rec / max(prec + rec, 1e-9))
    return cm, acc, float(np.mean(f1s)), f1s

def plot_confusion(cm, class_names, ax=None, title=""):
    if ax is None:
        _, ax = plt.subplots(figsize=(4.5, 4))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(class_names))); ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticks(range(len(class_names))); ax.set_yticklabels(class_names)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(title)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    return ax

### Data loaders & class weights
`class_weights` counter-balance the imbalance so the model can't cheat by always predicting
"no-damage".

In [ ]:
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

counts = np.bincount(train_ds.labels, minlength=4)
class_weights = torch.tensor(counts.sum() / (4 * np.maximum(counts, 1)), dtype=torch.float32)
print('class weights:', class_weights.numpy().round(2))

### 🧱 Train the from-scratch CNN

In [ ]:
t0 = time.time()
scratch = SiameseScratchCNN(NUM_CLASSES)
train_model(scratch, train_dl, val_dl, device, epochs=EPOCHS,
            class_weights=class_weights, ckpt_path=str(CKPT_DIR / 'scratch.pt'))
print(f'⏱️ scratch trained in {time.time()-t0:.1f}s')

### 🏋️ Train the pretrained ResNet18
First run downloads ImageNet weights (~45 MB) if internet is enabled; otherwise it uses random
init automatically.

In [ ]:
t0 = time.time()
pretrained = SiamesePretrainedCNN(NUM_CLASSES, freeze_until=6)
train_model(pretrained, train_dl, val_dl, device, epochs=EPOCHS,
            class_weights=class_weights, ckpt_path=str(CKPT_DIR / 'pretrained.pt'))
print(f'⏱️ pretrained trained in {time.time()-t0:.1f}s')

## 🎯 The key result: evaluate on a disaster nobody trained on
We compare both models on **SEEN** disasters (val) vs the **HELD-OUT** disaster (test). A big drop
on held-out = the transfer gap — the whole point of the project.

In [ ]:
import pandas as pd
rows = []; cms = {}
for name, model in [('scratch', scratch), ('pretrained', pretrained)]:
    for split, dl in [('SEEN (val)', val_dl), ('HELD-OUT (test)', test_dl)]:
        y, p = predict(model, dl, device)
        cm, acc, f1, _ = scores(y, p, NUM_CLASSES)
        cms[(name, split)] = cm
        rows.append({'model': name, 'split': split,
                     'accuracy': round(acc, 3), 'macro_F1': round(f1, 3)})
comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))
comparison

### Confusion matrices
Rows = true, cols = predicted. The diagonal is correct predictions.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 9))
for ax, ((name, split), cm) in zip(axes.ravel(), cms.items()):
    plot_confusion(cm, DAMAGE_CLASSES, ax=ax, title=f'{name} · {split}')
plt.tight_layout(); plt.show()

## 🗺️ Deployment: a damage map
Predict every building in an affected area and colour it green→red by damage level — the picture
responders would actually use. (Shown on synthetic data; for real data, see the note in the cell.)

In [ ]:
if USE_SYNTHETIC:
    GRID = 6; rng = np.random.default_rng(7)
    pres, posts, truth = [], [], []
    for _ in range(GRID*GRID):
        d = int(rng.choice(4, p=CLASS_P))
        a, b = make_pair(d, HELDOUT_DISASTER, rng)      # area hit by the held-out disaster
        pres.append(a); posts.append(b); truth.append(d)
    Xpre  = torch.stack([normalize_chw(a) for a in pres])
    Xpost = torch.stack([normalize_chw(b) for b in posts])
    with torch.no_grad():
        pred = pretrained(Xpre.to(device), Xpost.to(device)).argmax(1).cpu().numpy()
    COLOR = np.array([[.1,.7,.1],[.95,.85,.1],[.95,.5,.1],[.85,.1,.1]])
    P = PATCH_SIZE; mosaic = np.ones((GRID*P, GRID*P, 3), np.float32)
    for k, b in enumerate(posts):
        r, c = divmod(k, GRID); mosaic[r*P:(r+1)*P, c*P:(c+1)*P] = b
    fig, ax = plt.subplots(1, 3, figsize=(15, 5.5))
    ax[0].imshow(mosaic); ax[0].set_title('POST imagery (area)')
    ax[1].imshow(COLOR[np.array(truth).reshape(GRID, GRID)]); ax[1].set_title('True damage')
    ax[2].imshow(COLOR[pred.reshape(GRID, GRID)]); ax[2].set_title('Predicted damage')
    for a in ax: a.axis('off')
    plt.tight_layout(); plt.show()
    print('map accuracy:', round((pred == np.array(truth)).mean(), 2),
          '| green=ok, yellow=minor, orange=major, red=destroyed')
else:
    print('TODO(real data): pick a held-out xBD event tile, crop all its building patches,')
    print('predict with the better model, and draw each footprint coloured by predicted class.')

# Part 3 · Impact & Recovery Analysis  *(10 pts)*  — *open for the team* ✍️
This part is **freeform**. Goal: overlay our predicted damage with socioeconomic data and ask
whether damage/recovery is **unequally distributed** across communities.

**Steps to fill in**
1. Get socioeconomic data (income / population density / housing) for an affected region — e.g. US
   Census ACS for a US hurricane. Use the xBD **geotransforms** (`GEO_PATH`) for lat/lon.
2. Aggregate predicted damage per community (census tract / grid cell).
3. Correlate damage rate (and recovery, if multi-date) with income & density. Plot it + a map.
4. Write up: how rapid maps help responders; the risk of acting on *predicted* vs *verified*
   damage; limitations (label noise, cloud/smoke, imagery availability); and the ethics.

> 💡 Nothing here can break Parts 1–2 — this section only *reads* the model's predictions.

In [ ]:
# TODO(team): load socioeconomic table + predicted-damage-by-location, join, analyze, plot.
# import geopandas as gpd   # handy for spatial joins + choropleth maps
print('Part 3 is yours — see the steps above.')
print('geotransforms available at:', GEO_PATH)

## ✅ Done
You ran the full project: data → EDA → two models → held-out evaluation → damage map → (impact).

**To go from quick-test to real results:** attach the xBD dataset, set `USE_SYNTHETIC = False`,
confirm the path in the **Data path** cell, and Run All. Everything else stays the same.